In [2]:
import netCDF4 as nc
import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from tqdm import tqdm

/home/aidl/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


In [ ]:
feature_names = ['VPED','VSDX','VSDY',
    'VTM01_SW1', 'VTM01_SW2', 'VTM01_WW',
    'VTM02', 'VTM10', 'VTPK','VMDR_SW1', 'VCMX', 'VHM0',
    'VHM0_SW1', 'VHM0_SW2', 'VHM0_WW',
    'VMDR', 'VMDR_SW2', 'VMDR_WW', 'VMXL','TLA', 'TAUX', 'TAUY']

In [3]:
WAVE_VARS = [
    'VPED', 'VSDX', 'VSDY',
    'VTM01_SW1', 'VTM01_SW2', 'VTM01_WW',
    'VTM02', 'VTM10', 'VTPK'
]

WAVE_VARS1 = [
    'VMDR_SW1', 'VCMX', 'VHM0',
    'VHM0_SW1', 'VHM0_SW2', 'VHM0_WW',
    'VMDR', 'VMDR_SW2', 'VMDR_WW', 'VMXL'
]

EXTRA_VARS = ["TLA", "TAUX", "TAUY"]

In [4]:
ds  = nc.Dataset(r'new_dAtA/after_xl.nc')
ds1 = nc.Dataset(r'new_dAtA/till xl.nc')
ds3 = nc.Dataset(r'new_dAtA/tla_taux_tauy.nc')

getfattr: new_dAtA/till: No such file or directory
getfattr: xl.nc: No such file or directory


In [5]:
time_dim = ds.dimensions["time"].size
lat_dim  = ds.dimensions["latitude"].size
lon_dim  = ds.dimensions["longitude"].size

In [6]:
print(f"Time dim: {time_dim}, Lat dim: {lat_dim}, Lon dim: {lon_dim}")

Time dim: 35017, Lat dim: 162, Lon dim: 162


In [7]:
time_dim=10000


def load_time_slice(ds, var_list, t):
    data = []
    for v in var_list:
        data.append(ds.variables[v][t, :, :])
    return np.stack(data, axis=0)  # (C, H, W)

In [9]:
bathy_ds = nc.Dataset("new_dAtA/cmems_GEBCO_resampled_new.nc")
deptho = bathy_ds.variables["deptho"][:]  # (H, W)

assert deptho.shape == (lat_dim, lon_dim)

In [10]:
epsilon = 1e-6

def angular_difference(a, b):
    diff = np.abs(a - b)
    return np.minimum(diff, 360 - diff)

In [11]:
def compute_engineered_features(t):
    """
    Returns engineered feature tensor of shape (11, H, W)
    """

    # --- Directional & swell vars ---
    VMDR_SW1 = ds1.variables["VMDR_SW1"][t]
    VMDR_SW2 = ds1.variables["VMDR_SW2"][t]
    VMDR_WW  = ds1.variables["VMDR_WW"][t]

    VHM0_SW1 = ds1.variables["VHM0_SW1"][t]
    VHM0_SW2 = ds1.variables["VHM0_SW2"][t]

    # --- Stokes drift ---
    VSDX = ds.variables["VSDX"][t]
    VSDY = ds.variables["VSDY"][t]

    # --- Spectral ---
    VTPK  = ds.variables["VTPK"][t]
    VTM02 = ds.variables["VTM02"][t]

    # --- Dissipation ---
    TLA = ds3.variables["TLA"][t]

    # --- Wind stress ---
    TAUX = ds3.variables["TAUX"][t]
    TAUY = ds3.variables["TAUY"][t]

    # ==============================
    # Feature construction
    # ==============================

    delta_theta_SW = angular_difference(VMDR_SW1, VMDR_SW2)
    swell_energy_ratio = VHM0_SW2 / (VHM0_SW1 + epsilon)
    delta_theta_SW1_WW = angular_difference(VMDR_SW1, VMDR_WW)

    stokes_drift_mag = np.sqrt(VSDX**2 + VSDY**2)
    inv_stokes_drift = 1.0 / (stokes_drift_mag + epsilon)

    spectral_sharpness = VTPK / (VTM02 + epsilon)

    log_TLA = np.log(TLA + epsilon)
    TLA_x_swell_ratio = log_TLA * swell_energy_ratio
    TLA_x_delta_theta = log_TLA * delta_theta_SW

    TAU_mag = np.sqrt(TAUX**2 + TAUY**2)
    sin_TAU = np.sin(np.arctan2(TAUY, TAUX))
    cos_TAU = np.cos(np.arctan2(TAUY, TAUX))

    sin_VSD = np.sin(np.arctan2(VSDY, VSDX))
    cos_VSD = np.cos(np.arctan2(VSDY, VSDX))

    return np.stack([
        delta_theta_SW,
        swell_energy_ratio,
        delta_theta_SW1_WW,
        stokes_drift_mag,
        inv_stokes_drift,
        spectral_sharpness,
        log_TLA,
        TLA_x_swell_ratio,
        TLA_x_delta_theta,
        TAU_mag,
        sin_TAU,
        cos_TAU,
        sin_VSD,
        cos_VSD
    ], axis=0)

In [ ]:
def build_input_tensor(t):
    x0 = load_time_slice(ds,  WAVE_VARS,  t)    # (9, H, W)
    x1 = load_time_slice(ds1, WAVE_VARS1, t)    # (10, H, W)
    x2 = load_time_slice(ds3, EXTRA_VARS, t)    # (3, H, W)

    x_eng = compute_engineered_features(t)      # (12, H, W)
    bathy = deptho[None, :, :]                  # (1, H, W)

    x = np.concatenate([x0, x1, x2, x_eng, bathy], axis=0)
    # TOTAL CHANNELS = 9 + 10 + 3 + 14 + 1 = 37

    return torch.tensor(x, dtype=torch.float32)

In [21]:
class DeepLabEmbedding(nn.Module):
    def __init__(self, in_channels, embedding_dim=128):
        super().__init__()
        # --------------------------------------------------
        # 1×1 channel mixer (controls complexity & redundancy)
        # --------------------------------------------------
        self.channel_mixer = nn.Conv2d(
            in_channels=in_channels,
            out_channels=16,
            kernel_size=1,
            bias=False
        )
        # --------------------------------------------------
        # DeepLabV3++ backbone (encoder + ASPP only)
        # --------------------------------------------------
        self.backbone = smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None,   # IMPORTANT: not ImageNet
            in_channels=16,
            classes=1               # dummy head (unused)
        )
        self.encoder = self.backbone.encoder
        self.aspp = self.backbone.decoder.aspp
        # --------------------------------------------------
        # Global pooling → embedding
        # --------------------------------------------------
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.embedding_head = nn.Sequential(
            nn.Flatten(),                 # (B, 256)
            nn.Linear(256, embedding_dim),
            nn.LayerNorm(embedding_dim),  # ✅ batch-size-1 safe
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        """
        x: (B, C, H, W) where H,W are divisible by 16
        returns: (B, embedding_dim)
        """
        # Channel mixing
        x = self.channel_mixer(x)
        # DeepLab encoder
        features = self.encoder(x)[-1]
        # ASPP (multi-scale context)
        x = self.aspp(features)
        # Global pooling
        x = self.pool(x)
        # Embedding
        z = self.embedding_head(x)
        return z



In [22]:
model = DeepLabEmbedding(
    in_channels=37,
    embedding_dim=128
).to(device)

In [23]:

import torch.nn.functional as F

def pad_to_16(x):
    """
    x: Tensor of shape (C, H, W)
    Returns padded tensor with H, W divisible by 16
    """
    _, h, w = x.shape

    pad_h = (16 - h % 16) % 16
    pad_w = (16 - w % 16) % 16

    # Pad format: (left, right, top, bottom)
    x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")

    return x


In [26]:
from tqdm import tqdm

embeddings = []

# 1. Switch to Eval mode (Fixes the ValueError)
model.eval() 

with torch.no_grad():
    for t in tqdm(range(time_dim),
                  desc="DeepLabV3++ embedding",
                  unit="time-step",
                  dynamic_ncols=True):
        
        # 2. Prepare Input
        x_t = build_input_tensor(t)
        x_t = pad_to_16(x_t)
        
        # --- NEW: Sanitize Data ---
        # Replaces NaN with 0.0 and Infinity with large finite numbers
        x_t = torch.nan_to_num(x_t, nan=0.0, posinf=1e5, neginf=-1e5)
        
        x_t = x_t.unsqueeze(0).to(device)

        # 3. Forward Pass
        z_t = model(x_t)
        
        # 4. Store result
        embeddings.append(z_t.cpu())

DeepLabV3++ embedding:   0%|          | 0/10000 [00:00<?, ?time-step/s]/tmp/ipykernel_90069/3107830896.py:37: RuntimeWarning: invalid value encountered in sqrt
  stokes_drift_mag = np.sqrt(VSDX**2 + VSDY**2)
/tmp/ipykernel_90069/3107830896.py:42: RuntimeWarning: invalid value encountered in log
  log_TLA = np.log(TLA + epsilon)
/tmp/ipykernel_90069/3107830896.py:46: RuntimeWarning: invalid value encountered in sqrt
  TAU_mag = np.sqrt(TAUX**2 + TAUY**2)
DeepLabV3++ embedding:   0%|          | 44/10000 [00:11<24:33,  6.76time-step/s]  /tmp/ipykernel_90069/3107830896.py:42: RuntimeWarning: divide by zero encountered in log
  log_TLA = np.log(TLA + epsilon)
DeepLabV3++ embedding: 100%|██████████| 10000/10000 [25:12<00:00,  6.61time-step/s]  


In [27]:
Z = torch.cat(embeddings, dim=0)  # (T, 128)
np.save("deeplab_embeddings.npy", Z.numpy())